In [ ]:
!pip install -q transformers datasets evaluate bertviz accelerate --upgrade

from google.colab import drive
drive.mount('/content/drive')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.5/157.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.7/536.7 kB 18.2 MB/s eta 0:00:00
   ━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/47.6 MB 54.2 MB/s eta 0:00:01

In [ ]:
from datasets import load_dataset

ds = load_dataset("Aditya1010/17k-hotel-reviews-dataset")

label2id = {"NEGATIVE": 0, "POSITIVE": 1}
id2label = {0: "NEGATIVE", 1: "POSITIVE"}

def encode_labels(example):
    example["label"] = label2id[example["label"]]
    return example

ds = ds.map(encode_labels)

In [ ]:
from datasets import DatasetDict

TOTAL = 304
TRAIN_SIZE = 240
VAL_SIZE = 32
TEST_SIZE = 32

ds_small = ds["train"].shuffle(seed=42).select(range(TOTAL))

split = ds_small.train_test_split(test_size=TEST_SIZE + VAL_SIZE, seed=42)

temp_split = split["test"].train_test_split(test_size=TEST_SIZE, seed=42)

ds = DatasetDict({
    "train": split["train"],
    "validation": temp_split["train"],
    "test": temp_split["test"]
})

print(ds)

In [ ]:
#BERT
from transformers import AutoTokenizer

bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def tokenize_bert(examples):
    return bert_tokenizer(examples["review"], truncation=True, padding="max_length", max_length=128)

tokenized_ds_bert = ds.map(tokenize_bert, batched=True)
tokenized_ds_bert = tokenized_ds_bert.remove_columns(["review"])
tokenized_ds_bert.set_format("torch")


In [ ]:
#ROBERTA
roberta_tokenizer = AutoTokenizer.from_pretrained("roberta-base")

def tokenize_roberta(examples):
    return roberta_tokenizer(examples["review"], truncation=True, padding="max_length", max_length=128)

tokenized_ds_roberta = ds.map(tokenize_roberta, batched=True)
tokenized_ds_roberta = tokenized_ds_roberta.remove_columns(["review"])
tokenized_ds_roberta.set_format("torch")


In [ ]:
#BERT
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
import evaluate
import numpy as np

bert_model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-cased",
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

#Metrics
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="weighted")["f1"]
    }

data_collator = DataCollatorWithPadding(tokenizer=bert_tokenizer)

training_args_bert = TrainingArguments(
    output_dir="/content/drive/MyDrive/bert-sentiment-final1",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_steps=5,
    num_train_epochs=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
    seed=42
)

trainer_bert = Trainer(
    model=bert_model,
    args=training_args_bert,
    train_dataset=tokenized_ds_bert["train"],
    eval_dataset=tokenized_ds_bert["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer_bert.train()


In [ ]:
#ROBERTA
roberta_model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

data_collator_roberta = DataCollatorWithPadding(tokenizer=roberta_tokenizer)

training_args_roberta = TrainingArguments(
    output_dir="/content/drive/MyDrive/roberta-sentiment-final1",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_steps=5,
    num_train_epochs=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
    seed=42
)

trainer_roberta = Trainer(
    model=roberta_model,
    args=training_args_roberta,
    train_dataset=tokenized_ds_roberta["train"],
    eval_dataset=tokenized_ds_roberta["validation"],
    data_collator=data_collator_roberta,
    compute_metrics=compute_metrics
)

trainer_roberta.train()


In [ ]:
#BERT
preds_bert = trainer_bert.predict(tokenized_ds_bert["test"])
pred_labels_bert = np.argmax(preds_bert.predictions, axis=-1)

texts = ds["test"]["review"]
true_labels = ds["test"]["label"]

right_indices = [i for i, (p, t) in enumerate(zip(pred_labels_bert, true_labels)) if p==t]
wrong_indices = [i for i, (p, t) in enumerate(zip(pred_labels_bert, true_labels)) if p!=t]

print(f"BERT Correct: {len(right_indices)}, Wrong: {len(wrong_indices)}")


In [ ]:
#ROBERTA
preds_roberta = trainer_roberta.predict(tokenized_ds_roberta["test"])
pred_labels_roberta = np.argmax(preds_roberta.predictions, axis=-1)

right_indices_r = [i for i, (p, t) in enumerate(zip(pred_labels_roberta, true_labels)) if p==t]
wrong_indices_r = [i for i, (p, t) in enumerate(zip(pred_labels_roberta, true_labels)) if p!=t]

print(f"RoBERTa Correct: {len(right_indices_r)}, Wrong: {len(wrong_indices_r)}")


In [ ]:
#BERT
texts = ds["test"]["review"]
true_labels = ds["test"]["label"]

right_indices_bert = [i for i, (p, t) in enumerate(zip(pred_labels_bert, true_labels)) if p==t]
wrong_indices_bert = [i for i, (p, t) in enumerate(zip(pred_labels_bert, true_labels)) if p!=t]

print(f"BERT correct predictions: {len(right_indices_bert)}")
print(f"BERT wrong predictions: {len(wrong_indices_bert)}\n")

#First 10 correct
print("BERT: First 10 correct predictions\n")
for idx in right_indices_bert[:10]:
    print("Review:", texts[idx][:300], "...")
    print("Prediction:", id2label[pred_labels_bert[idx]])
    print("Gold:", id2label[true_labels[idx]])
    print("-"*80)

#First 10 wrong
print("BERT: First 10 wrong predictions\n")
for idx in wrong_indices_bert[:10]:
    print("Review:", texts[idx][:300], "...")
    print("Prediction:", id2label[pred_labels_bert[idx]])
    print("Gold:", id2label[true_labels[idx]])
    print("-"*80)

In [ ]:
#ROBERTA
right_indices_roberta = [i for i, (p, t) in enumerate(zip(pred_labels_roberta, true_labels)) if p==t]
wrong_indices_roberta = [i for i, (p, t) in enumerate(zip(pred_labels_roberta, true_labels)) if p!=t]

print(f"RoBERTa Correct predictions: {len(right_indices_roberta)}")
print(f"RoBERTa Wrong predictions: {len(wrong_indices_roberta)}\n")

#First 10 correct
print("RoBERTa: First 10 correct predictions\n")
for idx in right_indices_roberta[:10]:
    print("Review:", texts[idx][:300], "...")
    print("Prediction:", id2label[pred_labels_roberta[idx]])
    print("Gold:", id2label[true_labels[idx]])
    print("-"*80)

#First 10 wrong
print("RoBERTa: First 10 wrong predictions\n")
for idx in wrong_indices_roberta[:10]:
    print("Review:", texts[idx][:300], "...")
    print("Prediction:", id2label[pred_labels_roberta[idx]])
    print("Gold:", id2label[true_labels[idx]])
    print("-"*80)


In [ ]:
common_wrong = set(wrong_indices_bert).intersection(set(wrong_indices_roberta))

print(f"Common mistakes (BERT & RoBERTa): {len(common_wrong)}\n")

for idx in list(common_wrong)[:10]:
    print("Review:", texts[idx][:300], "...")
    print("BERT:", id2label[pred_labels_bert[idx]])
    print("RoBERTa:", id2label[pred_labels_roberta[idx]])
    print("Gold:", id2label[true_labels[idx]])
    print("-"*80)


In [ ]:
#BERT
import torch
import os
from torch.utils.tensorboard import SummaryWriter
from transformers import AutoModelForSequenceClassification

texts = list(ds["test"]["review"])
true_labels = list(ds["test"]["label"])

model_inputs = bert_tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

checkpoints_bert = [
    ("Epoch_1", "/content/drive/MyDrive/bert-sentiment-final1/checkpoint-15"),
    ("Epoch_3", "/content/drive/MyDrive/bert-sentiment-final1/checkpoint-45"),
    ("Epoch_5", "/content/drive/MyDrive/bert-sentiment-final1/checkpoint-75"),
]

for epoch_name, checkpoint_path in checkpoints_bert:
    print(f"Processing BERT {epoch_name}...")

    bert_model_vis = AutoModelForSequenceClassification.from_pretrained(
        checkpoint_path,
        num_labels=2,
        output_hidden_states=True
    )
    bert_model_vis.eval()

    with torch.no_grad():
        outputs = bert_model_vis(**model_inputs)

    num_examples = outputs.hidden_states[0].shape[0]
    texts_epoch = texts[:num_examples]
    labels_epoch = true_labels[:num_examples]

    n_layers = len(outputs.hidden_states)

    for layer in range(n_layers):
        layer_path = f"/content/drive/MyDrive/results_vis_bert_FINAL/{epoch_name}/layer_{layer}"
        os.makedirs(layer_path, exist_ok=True)

        tensors = []
        metadata = []

        for i in range(num_examples):
            cls_embedding = outputs.hidden_states[layer][i, 0]
            tensors.append(cls_embedding)

            clean_text = (
                texts_epoch[i]
                .replace("\n", " ")
                .replace("\r", " ")
                .replace("\t", " ")
            )
            metadata.append([clean_text, str(labels_epoch[i])])

        assert len(tensors) == len(metadata)

        writer = SummaryWriter(layer_path)
        writer.add_embedding(
            torch.stack(tensors),
            metadata=metadata,
            metadata_header=["Review", "Label"]
        )
        writer.close()

    print(f"BERT embeddings saved for {epoch_name}")

In [ ]:
#ROBERTA
import torch
import os
from torch.utils.tensorboard import SummaryWriter
from transformers import AutoModelForSequenceClassification

texts = list(ds["test"]["review"])
true_labels = list(ds["test"]["label"])

model_inputs_roberta = roberta_tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

checkpoints_roberta = [
    ("Epoch_1", "/content/drive/MyDrive/roberta-sentiment-final1/checkpoint-15"),
    ("Epoch_3", "/content/drive/MyDrive/roberta-sentiment-final1/checkpoint-45"),
    ("Epoch_5", "/content/drive/MyDrive/roberta-sentiment-final1/checkpoint-75"),
]

for epoch_name, checkpoint_path in checkpoints_roberta:
    print(f"Processing RoBERTa {epoch_name}...")

    roberta_model_vis = AutoModelForSequenceClassification.from_pretrained(
        checkpoint_path,
        num_labels=2,
        output_hidden_states=True
    )
    roberta_model_vis.eval()

    with torch.no_grad():
        outputs = roberta_model_vis(**model_inputs_roberta)

    num_examples = outputs.hidden_states[0].shape[0]
    texts_epoch = texts[:num_examples]
    labels_epoch = true_labels[:num_examples]

    n_layers = len(outputs.hidden_states)

    for layer in range(n_layers):
        layer_path = f"/content/drive/MyDrive/results_vis_roberta_FINAL/{epoch_name}/layer_{layer}"
        os.makedirs(layer_path, exist_ok=True)

        tensors = []
        metadata = []

        for i in range(num_examples):
            cls_embedding = outputs.hidden_states[layer][i, 0]
            tensors.append(cls_embedding)

            clean_text = (
                texts_epoch[i]
                .replace("\n", " ")
                .replace("\r", " ")
                .replace("\t", " ")
            )
            metadata.append([clean_text, str(labels_epoch[i])])

        assert len(tensors) == len(metadata)

        writer = SummaryWriter(layer_path)
        writer.add_embedding(
            torch.stack(tensors),
            metadata=metadata,
            metadata_header=["Review", "Label"]
        )
        writer.close()

    print(f"RoBERTa embeddings saved for {epoch_name}")